# Run 5: Improved U-Net v2 (Hybrid)

GatedConv Encoder + Bottleneck FG-BG Cross-Attention (8 heads) + Bilinear Attention Gates + Global/Local Color Affine Head.


In [ ]:
import os, sys, json, time, math, random, hashlib, zipfile
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from tqdm.auto import tqdm

# ==========================================
# 1. MOUNT DRIVE & GIẢI NÉN DỮ LIỆU
# ==========================================
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
if not PROJECT.exists():
    PROJECT = Path('/content/drive/MyDrive/ImageHarmonization')

DATA = Path('/content/iharmony4_256')
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
EXTS = {'.jpg', '.jpeg', '.png'}

for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

# ==========================================
# 2. CẤU HÌNH HUẤN LUYỆN
# ==========================================
CFG = dict(
    arch='ImprovedHarmonizationUNet_v2',
    size=256,
    base_channels=32,
    num_heads=8,
    bg_weight=0.1,
    batch_size=32,
    lr=3e-4,
    min_lr=1e-6,
    total_epochs=50,   # Giữ T_max=50 để đường cong Cosine LR chuẩn xác
    run_epochs=10,     # Đổi thành 50 nếu muốn chạy full luôn
    val_fraction=0.05,
    seed=42,
    num_workers=2,
)

RUN = PROJECT / 'runs/improved_unet_v2_hybrid_probe10ep_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Vui lòng bật GPU T4 trước khi chạy!'

def seed_all(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(CFG['seed'])

# ==========================================
# 3. CHIA TẬP DỮ LIỆU & XÁC MINH FINGERPRINT
# ==========================================
def make_index(folder: Path):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS}

records = {'train': [], 'val': [], 'test': []}
for subset in ARCHIVES:
    root = DATA / subset
    comp = make_index(root / 'composite_images')
    mask = make_index(root / 'masks')
    real = make_index(root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id = '_'.join(parts[:-2])
            mask_id = '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset,
                stem=stem,
                group=f'{subset}/{source_id}',
                full_id=f'{subset}/{stem}',
                comp=str(comp[stem]),
                mask=str(mask[mask_id]),
                real=str(real[source_id]),
            ))
        subsets[split] = rows

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [r['full_id'] for r in rows] for s, rows in records.items()}
FINGERPRINT = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
(RUN / 'split_manifest.json').write_text(json.dumps(split_ids))
print(f"Dataset sẵn sàng | Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")

class HarmonyDataset(Dataset):
    def __init__(self, rows, size=256, augment=False):
        self.rows, self.size, self.augment = rows, size, augment
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        target = (self.size, self.size)
        do_flip = self.augment and (random.random() < 0.5)
        imgs = []
        for k, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[k]) as im:
                im = im.convert(mode)
                if im.size != target:
                    im = im.resize(target, Image.Resampling.NEAREST if k == 'mask' else Image.Resampling.BILINEAR)
                if do_flip:
                    im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                imgs.append(pil_to_tensor(im).float() / 255.)
        c, m, y = imgs
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        return c, m, y, r['subset']

# ==========================================
# 4. KIẾN TRÚC IMPROVED HARMONIZATION U-NET V2
# ==========================================
class GatedConv2d(nn.Module):
    """[1] Gated Convolution với GroupNorm: học trọng số tin cậy theo mask và bảo toàn thang độ sáng."""
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding, bias=False)
        self.norm = nn.GroupNorm(num_groups=math.gcd(8, out_ch), num_channels=out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat, gate = self.conv(x).chunk(2, dim=1)
        return self.act(self.norm(feat)) * torch.sigmoid(gate)


class GatedDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            GatedConv2d(in_ch, out_ch),
            GatedConv2d(out_ch, out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class StandardDoubleConv(nn.Module):
    """Decoder DoubleConv dùng GroupNorm đồng nhất."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        num_groups = math.gcd(8, out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BilinearUp(nn.Module):
    """Upsample Bilinear + Conv 3x3 thay cho ConvTranspose2d để triệt tiêu nhiễu bàn cờ."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.up(x)


class AttentionGate(nn.Module):
    """[2] Attention-Gated Skip: dùng đặc trưng Decoder lọc bỏ màu sai từ Encoder skip."""
    def __init__(self, g_ch: int, x_ch: int):
        super().__init__()
        inter_ch = max(x_ch // 2, 16)
        self.W_g = nn.Conv2d(g_ch, inter_ch, kernel_size=1, bias=True)
        self.W_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        alpha = self.psi(self.W_g(g) + self.W_x(x))
        return x * alpha


class BottleneckFGBGCrossAttention(nn.Module):
    """[3] Cross-Attention + FFN tại đáy U-Net: pixel Foreground lấy mẫu ánh sáng từ toàn bộ Background."""
    def __init__(self, channels: int = 512, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.norm_q = nn.LayerNorm(channels)
        self.norm_kv = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.gamma1 = nn.Parameter(torch.zeros(1))
        self.gamma2 = nn.Parameter(torch.zeros(1))

    def forward(self, feat: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, C, H, W = feat.shape
        m_low = F.adaptive_avg_pool2d(mask, (H, W))
        m_flat = m_low.view(B, H * W)

        tokens = feat.flatten(2).transpose(1, 2)  # [B, N, C]
        q = self.norm_q(tokens)
        kv = self.norm_kv(tokens)

        key_padding_mask = m_flat > 0.5
        all_fg = key_padding_mask.all(dim=1)
        if all_fg.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[all_fg] = False

        attn_out, _ = self.attn(
            query=q,
            key=kv,
            value=kv,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        fg_gate_tokens = (m_flat > 0.2).unsqueeze(-1).to(feat.dtype)
        tokens = tokens + self.gamma1 * (attn_out * fg_gate_tokens)
        tokens = tokens + self.gamma2 * (self.ffn(self.norm_ffn(tokens)) * fg_gate_tokens)

        return tokens.transpose(1, 2).view(B, C, H, W)


class GlobalLocalColorAffineHead(nn.Module):
    """[4] Dự đoán biến đổi tuyến tính màu (Global 3x3 + Local 3x3 + bias) thay vì vẽ lại pixel RGB."""
    def __init__(self, in_ch: int):
        super().__init__()
        self.local_conv = nn.Conv2d(in_ch, 12, kernel_size=3, padding=1)
        self.global_fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_ch, 12),
        )
        nn.init.zeros_(self.local_conv.weight)
        nn.init.zeros_(self.local_conv.bias)
        nn.init.zeros_(self.global_fc[2].weight)
        nn.init.zeros_(self.global_fc[2].bias)

    def forward(self, feat: torch.Tensor, comp: torch.Tensor) -> torch.Tensor:
        B, _, H, W = feat.shape
        params = self.local_conv(feat) + self.global_fc(feat).view(B, 12, 1, 1)
        dW = params[:, :9].view(B, 3, 3, H, W)
        db = params[:, 9:].view(B, 3, H, W)

        eye = torch.eye(3, device=feat.device, dtype=feat.dtype).view(1, 3, 3, 1, 1)
        pred = torch.einsum("bijhw,bjhw->bihw", (eye + dW).float(), comp.float()) + db.float()
        return torch.clamp(pred, 0.0, 1.0).to(feat.dtype)


class ImprovedHarmonizationUNet(nn.Module):
    def __init__(self, base_channels: int = 32, num_heads: int = 8):
        super().__init__()
        c1, c2, c3, c4, c5 = (
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
            base_channels * 16,
        )

        self.enc1 = GatedDoubleConv(4, c1)
        self.enc2 = GatedDoubleConv(c1, c2)
        self.enc3 = GatedDoubleConv(c2, c3)
        self.enc4 = GatedDoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = GatedDoubleConv(c4, c5)
        self.fg_bg_attn = BottleneckFGBGCrossAttention(channels=c5, num_heads=num_heads)

        self.up4 = BilinearUp(c5, c4)
        self.att4 = AttentionGate(g_ch=c4, x_ch=c4)
        self.dec4 = StandardDoubleConv(c4 * 2, c4)

        self.up3 = BilinearUp(c4, c3)
        self.att3 = AttentionGate(g_ch=c3, x_ch=c3)
        self.dec3 = StandardDoubleConv(c3 * 2, c3)

        self.up2 = BilinearUp(c3, c2)
        self.att2 = AttentionGate(g_ch=c2, x_ch=c2)
        self.dec2 = StandardDoubleConv(c2 * 2, c2)

        self.up1 = BilinearUp(c2, c1)
        self.att1 = AttentionGate(g_ch=c1, x_ch=c1)
        self.dec1 = StandardDoubleConv(c1 * 2, c1)

        self.head = GlobalLocalColorAffineHead(c1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        comp = x[:, :3]
        mask = x[:, 3:4]

        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        bot = self.bottleneck(self.pool(s4))
        bot = self.fg_bg_attn(bot, mask)

        g4 = self.up4(bot)
        d4 = self.dec4(torch.cat([g4, self.att4(g4, s4)], dim=1))

        g3 = self.up3(d4)
        d3 = self.dec3(torch.cat([g3, self.att3(g3, s3)], dim=1))

        g2 = self.up2(d3)
        d2 = self.dec2(torch.cat([g2, self.att2(g2, s2)], dim=1))

        g1 = self.up1(d2)
        d1 = self.dec1(torch.cat([g1, self.att1(g1, s1)], dim=1))

        return self.head(d1, comp)


def compose(pred: torch.Tensor, comp: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return mask * pred + (1.0 - mask) * comp


def hybrid_harmonization_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    bg_weight: float = 0.1,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, dict[str, float]]:
    fg_diff = torch.abs(pred - target) * mask
    fg_loss = (fg_diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    bg_mask = 1.0 - mask
    bg_diff = torch.abs(pred - target) * bg_mask
    bg_loss = (bg_diff.sum(dim=(1, 2, 3)) / (3.0 * bg_mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    total_loss = fg_loss + bg_weight * bg_loss
    return total_loss, {
        "train_fL1": fg_loss.item(),
        "train_bgL1": bg_loss.item(),
    }


# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN, RESUME & ĐÁNH GIÁ
# ==========================================
model = ImprovedHarmonizationUNet(base_channels=CFG['base_channels'], num_heads=CFG['num_heads']).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Khởi tạo {CFG['arch']} thành công | Tổng tham số: {total_params:.2f}M")

train_loader = DataLoader(
    HarmonyDataset(records['train'], CFG['size'], augment=True),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    HarmonyDataset(records['val'], CFG['size'], augment=False),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
)

opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['total_epochs'], eta_min=CFG['min_lr'])
scaler = torch.amp.GradScaler('cuda')

@torch.inference_mode()
def evaluate():
    model.eval()
    sums = defaultdict(Counter)
    preview = None
    for c, m, y, subsets in val_loader:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], dim=1))
        raw = raw.float()
        out = compose(raw, c, m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3.0 * m.sum((1, 2, 3)).clamp_min(1.0))
        l1 = ((out - y).abs() * m).sum((1, 2, 3)) / (3.0 * m.sum((1, 2, 3)).clamp_min(1.0))

        vals = dict(
            raw_L1=l1 * 255.0,
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        for i, s in enumerate(subsets):
            for label in ('all', s):
                sums[label]['n'] += 1
                for k, v in vals.items():
                    sums[label][k] += v[i].item()
        if preview is None:
            preview = torch.stack([c[:6], m[:6].expand_as(c[:6]), raw[:6], out[:6], y[:6]], 1).flatten(0, 1).cpu()
    metrics = {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}
    return metrics, preview

history = []
best_fmse = float('inf')
start_epoch = 1

last_ck_path = RUN / 'last.pt'
if last_ck_path.exists():
    ck = torch.load(last_ck_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    best_fmse = ck['best_fmse']
    if (RUN / 'history.json').exists():
        history = json.loads((RUN / 'history.json').read_text())
    print(f"Đã khôi phục checkpoint tại Epoch {ck['epoch']} | Best fMSE: {best_fmse:.2f}")

for epoch in range(start_epoch, CFG['run_epochs'] + 1):
    t0 = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    model.train()
    tr_total, tr_fl1, tr_bgl1, n_steps = 0.0, 0.0, 0.0, 0

    for c, m, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{CFG['run_epochs']}", leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], dim=1))
            loss, logs = hybrid_harmonization_loss(raw.float(), y, m, bg_weight=CFG['bg_weight'])
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        tr_total += loss.item()
        tr_fl1 += logs['train_fL1']
        tr_bgl1 += logs['train_bgL1']
        n_steps += 1

    sched.step()
    val_metrics, preview = evaluate()
    sec = time.perf_counter() - t0
    vram = torch.cuda.max_memory_allocated() / (1024**3)

    rec = dict(
        epoch=epoch,
        train_loss=tr_total / n_steps,
        train_fL1=tr_fl1 / n_steps,
        train_bgL1=tr_bgl1 / n_steps,
        val=val_metrics,
        lr=opt.param_groups[0]['lr'],
        seconds=sec,
        peak_VRAM_GB=vram,
    )
    history.append(rec)
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    save_image(preview, str(RUN / f'val_preview_ep{epoch:02d}.png'), nrow=5)

    cur_fmse = val_metrics['all']['fMSE']
    if cur_fmse < best_fmse:
        best_fmse = cur_fmse
        torch.save(dict(
            epoch=epoch, cfg=CFG, fingerprint=FINGERPRINT, best_fmse=best_fmse,
            model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
            val=val_metrics
        ), RUN / 'best.pt')

    torch.save(dict(
        epoch=epoch, cfg=CFG, fingerprint=FINGERPRINT, best_fmse=best_fmse,
        model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
        scaler=scaler.state_dict(), val=val_metrics
    ), last_ck_path)

    print(f"[Ep {epoch:02d}] fL1: {rec['train_fL1']:.4f} | bgL1: {rec['train_bgL1']:.4f} | "
          f"val_fMSE: {cur_fmse:.2f} | val_PSNR: {val_metrics['all']['PSNR']:.2f} | "
          f"best: {best_fmse:.2f} | {sec:.0f}s | {vram:.2f}GB")

Mounted at /content/drive
Đang giải nén HCOCO_256.zip...


In [ ]:
import os, sys, json, time, math, random, hashlib, zipfile
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from tqdm.auto import tqdm

# ==========================================
# 1. MOUNT DRIVE & GIẢI NÉN DỮ LIỆU
# ==========================================
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
if not PROJECT.exists():
    PROJECT = Path('/content/drive/MyDrive/ImageHarmonization')

DATA = Path('/content/iharmony4_256')
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
EXTS = {'.jpg', '.jpeg', '.png'}

for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

# ==========================================
# 2. CẤU HÌNH HUẤN LUYỆN (ĐÃ ĐẶT 50 EPOCHS)
# ==========================================
CFG = dict(
    arch='ImprovedHarmonizationUNet_v2',
    size=256,
    base_channels=32,
    num_heads=8,
    bg_weight=0.1,
    batch_size=32,
    lr=3e-4,
    min_lr=1e-6,
    total_epochs=50,
    run_epochs=50,
    val_fraction=0.05,
    seed=42,
    num_workers=2,
)

RUN = PROJECT / 'runs/improved_unet_v2_hybrid_probe10ep_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Vui lòng bật GPU trước khi chạy!'

def seed_all(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(CFG['seed'])

# ==========================================
# 3. CHIA TẬP DỮ LIỆU & XÁC MINH FINGERPRINT
# ==========================================
def make_index(folder: Path):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS}

records = {'train': [], 'val': [], 'test': []}
for subset in ARCHIVES:
    root = DATA / subset
    comp = make_index(root / 'composite_images')
    mask = make_index(root / 'masks')
    real = make_index(root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id = '_'.join(parts[:-2])
            mask_id = '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset,
                stem=stem,
                group=f'{subset}/{source_id}',
                full_id=f'{subset}/{stem}',
                comp=str(comp[stem]),
                mask=str(mask[mask_id]),
                real=str(real[source_id]),
            ))
        subsets[split] = rows

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [r['full_id'] for r in rows] for s, rows in records.items()}
FINGERPRINT = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
(RUN / 'split_manifest.json').write_text(json.dumps(split_ids))
print(f"Dataset sẵn sàng | Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")

class HarmonyDataset(Dataset):
    def __init__(self, rows, size=256, augment=False):
        self.rows, self.size, self.augment = rows, size, augment
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        target = (self.size, self.size)
        do_flip = self.augment and (random.random() < 0.5)
        imgs = []
        for k, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[k]) as im:
                im = im.convert(mode)
                if im.size != target:
                    im = im.resize(target, Image.Resampling.NEAREST if k == 'mask' else Image.Resampling.BILINEAR)
                if do_flip:
                    im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                imgs.append(pil_to_tensor(im).float() / 255.)
        c, m, y = imgs
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        return c, m, y, r['subset']

# ==========================================
# 4. KIẾN TRÚC IMPROVED HARMONIZATION U-NET V2
# ==========================================
class GatedConv2d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding, bias=False)
        self.norm = nn.GroupNorm(num_groups=math.gcd(8, out_ch), num_channels=out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat, gate = self.conv(x).chunk(2, dim=1)
        return self.act(self.norm(feat)) * torch.sigmoid(gate)


class GatedDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            GatedConv2d(in_ch, out_ch),
            GatedConv2d(out_ch, out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class StandardDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        num_groups = math.gcd(8, out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BilinearUp(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.up(x)


class AttentionGate(nn.Module):
    def __init__(self, g_ch: int, x_ch: int):
        super().__init__()
        inter_ch = max(x_ch // 2, 16)
        self.W_g = nn.Conv2d(g_ch, inter_ch, kernel_size=1, bias=True)
        self.W_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        alpha = self.psi(self.W_g(g) + self.W_x(x))
        return x * alpha


class BottleneckFGBGCrossAttention(nn.Module):
    def __init__(self, channels: int = 512, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.norm_q = nn.LayerNorm(channels)
        self.norm_kv = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.gamma1 = nn.Parameter(torch.zeros(1))
        self.gamma2 = nn.Parameter(torch.zeros(1))

    def forward(self, feat: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, C, H, W = feat.shape
        m_low = F.adaptive_avg_pool2d(mask, (H, W))
        m_flat = m_low.view(B, H * W)

        tokens = feat.flatten(2).transpose(1, 2)
        q = self.norm_q(tokens)
        kv = self.norm_kv(tokens)

        key_padding_mask = m_flat > 0.5
        all_fg = key_padding_mask.all(dim=1)
        if all_fg.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[all_fg] = False

        attn_out, _ = self.attn(
            query=q,
            key=kv,
            value=kv,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        fg_gate_tokens = (m_flat > 0.2).unsqueeze(-1).to(feat.dtype)
        tokens = tokens + self.gamma1 * (attn_out * fg_gate_tokens)
        tokens = tokens + self.gamma2 * (self.ffn(self.norm_ffn(tokens)) * fg_gate_tokens)

        return tokens.transpose(1, 2).view(B, C, H, W)


class GlobalLocalColorAffineHead(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.local_conv = nn.Conv2d(in_ch, 12, kernel_size=3, padding=1)
        self.global_fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_ch, 12),
        )
        nn.init.zeros_(self.local_conv.weight)
        nn.init.zeros_(self.local_conv.bias)
        nn.init.zeros_(self.global_fc[2].weight)
        nn.init.zeros_(self.global_fc[2].bias)

    def forward(self, feat: torch.Tensor, comp: torch.Tensor) -> torch.Tensor:
        B, _, H, W = feat.shape
        params = self.local_conv(feat) + self.global_fc(feat).view(B, 12, 1, 1)
        dW = params[:, :9].view(B, 3, 3, H, W)
        db = params[:, 9:].view(B, 3, H, W)

        eye = torch.eye(3, device=feat.device, dtype=feat.dtype).view(1, 3, 3, 1, 1)
        pred = torch.einsum("bijhw,bjhw->bihw", (eye + dW).float(), comp.float()) + db.float()
        return torch.clamp(pred, 0.0, 1.0).to(feat.dtype)


class ImprovedHarmonizationUNet(nn.Module):
    def __init__(self, base_channels: int = 32, num_heads: int = 8):
        super().__init__()
        c1, c2, c3, c4, c5 = (
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
            base_channels * 16,
        )

        self.enc1 = GatedDoubleConv(4, c1)
        self.enc2 = GatedDoubleConv(c1, c2)
        self.enc3 = GatedDoubleConv(c2, c3)
        self.enc4 = GatedDoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = GatedDoubleConv(c4, c5)
        self.fg_bg_attn = BottleneckFGBGCrossAttention(channels=c5, num_heads=num_heads)

        self.up4 = BilinearUp(c5, c4)
        self.att4 = AttentionGate(g_ch=c4, x_ch=c4)
        self.dec4 = StandardDoubleConv(c4 * 2, c4)

        self.up3 = BilinearUp(c4, c3)
        self.att3 = AttentionGate(g_ch=c3, x_ch=c3)
        self.dec3 = StandardDoubleConv(c3 * 2, c3)

        self.up2 = BilinearUp(c3, c2)
        self.att2 = AttentionGate(g_ch=c2, x_ch=c2)
        self.dec2 = StandardDoubleConv(c2 * 2, c2)

        self.up1 = BilinearUp(c2, c1)
        self.att1 = AttentionGate(g_ch=c1, x_ch=c1)
        self.dec1 = StandardDoubleConv(c1 * 2, c1)

        self.head = GlobalLocalColorAffineHead(c1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        comp = x[:, :3]
        mask = x[:, 3:4]

        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        bot = self.bottleneck(self.pool(s4))
        bot = self.fg_bg_attn(bot, mask)

        g4 = self.up4(bot)
        d4 = self.dec4(torch.cat([g4, self.att4(g4, s4)], dim=1))

        g3 = self.up3(d4)
        d3 = self.dec3(torch.cat([g3, self.att3(g3, s3)], dim=1))

        g2 = self.up2(d3)
        d2 = self.dec2(torch.cat([g2, self.att2(g2, s2)], dim=1))

        g1 = self.up1(d2)
        d1 = self.dec1(torch.cat([g1, self.att1(g1, s1)], dim=1))

        return self.head(d1, comp)


def compose(pred: torch.Tensor, comp: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return mask * pred + (1.0 - mask) * comp


def hybrid_harmonization_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    bg_weight: float = 0.1,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, dict[str, float]]:
    fg_diff = torch.abs(pred - target) * mask
    fg_loss = (fg_diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    bg_mask = 1.0 - mask
    bg_diff = torch.abs(pred - target) * bg_mask
    bg_loss = (bg_diff.sum(dim=(1, 2, 3)) / (3.0 * bg_mask.sum(dim=(1, 2, 3)).clamp_min(eps))).mean()

    total_loss = fg_loss + bg_weight * bg_loss
    return total_loss, {
        "train_fL1": fg_loss.item(),
        "train_bgL1": bg_loss.item(),
    }


# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN, AUTO-RESUME & ĐÁNH GIÁ
# ==========================================
model = ImprovedHarmonizationUNet(base_channels=CFG['base_channels'], num_heads=CFG['num_heads']).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Khởi tạo {CFG['arch']} thành công | Tổng tham số: {total_params:.2f}M")

train_loader = DataLoader(
    HarmonyDataset(records['train'], CFG['size'], augment=True),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    HarmonyDataset(records['val'], CFG['size'], augment=False),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
)

opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['total_epochs'], eta_min=CFG['min_lr'])
scaler = torch.amp.GradScaler('cuda')

@torch.inference_mode()
def evaluate():
    model.eval()
    sums = defaultdict(Counter)
    preview = None
    for c, m, y, subsets in val_loader:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], dim=1))
        raw = raw.float()
        out = compose(raw, c, m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3.0 * m.sum((1, 2, 3)).clamp_min(1.0))
        l1 = ((out - y).abs() * m).sum((1, 2, 3)) / (3.0 * m.sum((1, 2, 3)).clamp_min(1.0))

        vals = dict(
            raw_L1=l1 * 255.0,
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        for i, s in enumerate(subsets):
            for label in ('all', s):
                sums[label]['n'] += 1
                for k, v in vals.items():
                    sums[label][k] += v[i].item()
        if preview is None:
            preview = torch.stack([c[:6], m[:6].expand_as(c[:6]), raw[:6], out[:6], y[:6]], 1).flatten(0, 1).cpu()
    metrics = {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}
    return metrics, preview

history = []
best_fmse = float('inf')
start_epoch = 1

last_ck_path = RUN / 'last.pt'
if last_ck_path.exists():
    ck = torch.load(last_ck_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    best_fmse = ck['best_fmse']
    if (RUN / 'history.json').exists():
        history = json.loads((RUN / 'history.json').read_text())
    print(f"Đã khôi phục checkpoint tại Epoch {ck['epoch']} | Best fMSE: {best_fmse:.2f}")

for epoch in range(start_epoch, CFG['run_epochs'] + 1):
    t0 = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    model.train()
    tr_total, tr_fl1, tr_bgl1, n_steps = 0.0, 0.0, 0.0, 0

    for c, m, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{CFG['run_epochs']}", leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], dim=1))
            loss, logs = hybrid_harmonization_loss(raw.float(), y, m, bg_weight=CFG['bg_weight'])
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        tr_total += loss.item()
        tr_fl1 += logs['train_fL1']
        tr_bgl1 += logs['train_bgL1']
        n_steps += 1

    sched.step()
    val_metrics, preview = evaluate()
    sec = time.perf_counter() - t0
    vram = torch.cuda.max_memory_allocated() / (1024**3)

    rec = dict(
        epoch=epoch,
        train_loss=tr_total / n_steps,
        train_fL1=tr_fl1 / n_steps,
        train_bgL1=tr_bgl1 / n_steps,
        val=val_metrics,
        lr=opt.param_groups[0]['lr'],
        seconds=sec,
        peak_VRAM_GB=vram,
    )
    history.append(rec)
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    save_image(preview, str(RUN / f'val_preview_ep{epoch:02d}.png'), nrow=5)

    cur_fmse = val_metrics['all']['fMSE']
    if cur_fmse < best_fmse:
        best_fmse = cur_fmse
        torch.save(dict(
            epoch=epoch, cfg=CFG, fingerprint=FINGERPRINT, best_fmse=best_fmse,
            model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
            val=val_metrics
        ), RUN / 'best.pt')

    torch.save(dict(
        epoch=epoch, cfg=CFG, fingerprint=FINGERPRINT, best_fmse=best_fmse,
        model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
        scaler=scaler.state_dict(), val=val_metrics
    ), last_ck_path)

    print(f"[Ep {epoch:02d}] fL1: {rec['train_fL1']:.4f} | bgL1: {rec['train_bgL1']:.4f} | "
          f"val_fMSE: {cur_fmse:.2f} | val_PSNR: {val_metrics['all']['PSNR']:.2f} | "
          f"best: {best_fmse:.2f} | {sec:.0f}s | {vram:.2f}GB")

Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dataset sẵn sàng | Train: 62463 | Val: 3279 | Test: 7404
Khởi tạo ImprovedHarmonizationUNet_v2 thành công | Tổng tham số: 15.54M
Đã khôi phục checkpoint tại Epoch 48 | Best fMSE: 382.23


[Ep 49] fL1: 0.0377 | bgL1: 0.0044 | val_fMSE: 404.24 | val_PSNR: 36.11 | best: 382.23 | 1616s | 8.27GB


[Ep 50] fL1: 0.0376 | bgL1: 0.0044 | val_fMSE: 404.45 | val_PSNR: 36.10 | best: 382.23 | 1611s | 8.27GB
